In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_supplier, q15
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15" #"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # Use GPU parse_dates to avoid post-read conversions
    df = pd.read_csv(
        data_path,
        sep="|",
        storage_options=storage_options,
        parse_dates=["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"]
    )
    print(df.columns)
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:
### cell 1 ###


def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [6]:
%%time
### cell 2 ###

# Optimized GPU workflow using cudf.pandas
# Define date bounds as strings to push comparisons to the GPU
date_start = "1996-01-01"
date_end = "1996-04-01"

# Filter, compute revenue parts, aggregate and pick top supplier in one pipeline
top_revenue = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= date_start) & (lineitem.L_SHIPDATE < date_end),
        ["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
    ]
    .assign(REVENUE_PARTS=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .groupby("L_SUPPKEY")["REVENUE_PARTS"].sum()
    .reset_index()
    .nlargest(1, "REVENUE_PARTS")
    .rename(columns={"L_SUPPKEY": "S_SUPPKEY", "REVENUE_PARTS": "TOTAL_REVENUE"})
)

# Join with supplier details and select final columns
total = (
    top_revenue
    .merge(
        supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]],
        on="S_SUPPKEY"
    )
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)


CPU times: user 693 ms, sys: 283 ms, total: 976 ms
Wall time: 1.24 s


In [7]:
%%time

lineitem_filtered = lineitem[
    (lineitem["L_SHIPDATE"] >= pd.Timestamp("1996-01-01"))
    & (
        lineitem["L_SHIPDATE"]
        < (pd.Timestamp("1996-01-01") + pd.DateOffset(months=3))
    )
]
lineitem_filtered["REVENUE_PARTS"] = lineitem_filtered["L_EXTENDEDPRICE"] * (
    1.0 - lineitem_filtered["L_DISCOUNT"]
)
lineitem_filtered = lineitem_filtered.loc[:, ["L_SUPPKEY", "REVENUE_PARTS"]]
revenue_table = (
    lineitem_filtered.groupby("L_SUPPKEY", as_index=False, sort=False)
    .agg(TOTAL_REVENUE=pd.NamedAgg(column="REVENUE_PARTS", aggfunc="sum"))
    .rename(columns={"L_SUPPKEY": "SUPPLIER_NO"})
)
max_revenue = revenue_table["TOTAL_REVENUE"].max()
revenue_table = revenue_table[revenue_table["TOTAL_REVENUE"] == max_revenue]
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]]
total = supplier_filtered.merge(
    revenue_table, left_on="S_SUPPKEY", right_on="SUPPLIER_NO", how="inner"
)
total_orig = total.loc[
    :, ["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]
]




CPU times: user 1.35 s, sys: 841 ms, total: 2.19 s
Wall time: 2.07 s
